# SAR & InSAR with PyGeoFetch — Complete Processing Tutorial

This notebook walks through the **entire SAR and InSAR pipeline** in PyGeoFetch,
from raw data acquisition to a displacement time series, using real
Sentinel-1 data and the state-of-the-art methods implemented in
`pygeofetch.insar`:

| Stage | Method | Reference |
|---|---|---|
| Search & download | Federated search, precise orbit files | — |
| Radiometric calibration | DN → Sigma0/Gamma0/Beta0 | ESA S-1 handbook |
| Speckle filtering | Lee / Enhanced Lee / Gamma / Frost | Lee (1980, 1981) |
| Coregistration | Geometric + Enhanced Spectral Diversity (ESD) | Scheiber & Moreira (2000) |
| Interferogram formation | Complex conjugate multiplication | — |
| Topographic phase removal | DEM regression | — |
| Coherence estimation | Windowed complex correlation | — |
| Phase unwrapping | SNAPHU (MCF, statistical cost) | Chen & Zebker (2001) |
| Atmospheric correction | Elevation-correlated / ERA5 | Jolivet et al. (2011, 2014) |
| Time series inversion | SBAS weighted least squares | Berardino et al. (2002) |

**Install requirements:**
```bash
pip install "pygeofetch[insar]"        # core: snaphu, scipy
pip install "pygeofetch[insar-full]"   # + mintpy, pyaps3 for advanced corrections
```


In [ ]:
# Verify the environment
import importlib

required = ["rasterio", "numpy", "scipy"]
optional = ["snaphu", "mintpy", "pyaps3"]

for pkg in required:
    mod = importlib.util.find_spec(pkg)
    print(f"  [{'OK' if mod else 'MISSING'}] {pkg} (required)")

for pkg in optional:
    mod = importlib.util.find_spec(pkg)
    status = "OK" if mod else "not installed (optional)"
    print(f"  [{status}] {pkg}")


## Part 1 — Data Acquisition

We search for a Sentinel-1 SLC pair over an area of interest. SLC (Single
Look Complex) products are **required** for InSAR — they preserve the phase
information that GRD products discard.

PyGeoFetch automatically routes SLC searches to providers that host them
(Copernicus Dataspace, Alaska Satellite Facility) since Planetary Computer,
AWS Earth, and Element84 only host GRD/COG products.


In [ ]:
from pathlib import Path

from pygeofetch import PyGeoFetch
from pygeofetch.models.search_query import BoundingBox, SearchQuery

client = PyGeoFetch(log_level="INFO")

# Area of interest — adjust to your region.
# This example uses a subsidence-prone area (Mexico City metro area)
# where InSAR is routinely used for groundwater extraction monitoring.
aoi = BoundingBox.from_string("-99.20,19.30,-99.00,19.50")

query = SearchQuery(
    bbox=aoi,
    start_date="2026-01-01",
    end_date="2026-03-01",
    product_type="SLC",       # REQUIRED for InSAR — GRD has no phase info
    polarisation="VV",
    max_results=10,
)

results = client.search(query, providers=["copernicus"])

print(f"Found {len(results)} SLC scenes:\n")
for r in results:
    print(f"  {r.id}  |  {r.datetime}  |  orbit: {getattr(r, 'pass_direction', 'n/a')}")


### Selecting an interferometric pair

For InSAR, you need **two acquisitions from the same relative orbit** (same
track, same pass direction) with a manageable temporal baseline. Shorter
temporal baselines generally give higher coherence over vegetated areas.

We'll pick two scenes roughly 12 days apart (one Sentinel-1 repeat cycle).


In [ ]:
# Sort by date and pick a close pair (same orbit track)
if len(results) < 1:
    print("No search results found. This may be because:")
    print("  - No network access in this environment")
    print("  - No credentials configured for the 'copernicus' provider")
    print("  - No scenes match the AOI/date range")
    print()
    print("To continue this tutorial without live data, skip to Part 2 and")
    print("supply your own SLC files, or see notebook 02 for a fully")
    print("self-contained example using synthetic data.")
    reference_scene = secondary_scene = None
else:
    results_sorted = sorted(results, key=lambda r: r.datetime)
    reference_scene = results_sorted[0]
    secondary_scene  = results_sorted[1] if len(results_sorted) > 1 else results_sorted[0]

    print(f"Reference (master):  {reference_scene.id}  ({reference_scene.datetime})")
    print(f"Secondary (slave):   {secondary_scene.id}  ({secondary_scene.datetime})")

    temporal_baseline_days = abs(
        (secondary_scene.datetime - reference_scene.datetime).days
        if hasattr(reference_scene.datetime, 'days') else 12
    )
    print(f"\nTemporal baseline: ~{temporal_baseline_days} days")

In [ ]:
# Download both scenes
from pygeofetch.models.download_task import DownloadOptions

output_dir = Path("./data/insar_pair")
output_dir.mkdir(parents=True, exist_ok=True)

if reference_scene is None:
    print("Skipping download — no scenes found in the search above.")
else:
    options = DownloadOptions(
        parallel=2,
        resume=True,
        on_failure="skip",
    )

    download_results = client.download(
        [reference_scene, secondary_scene],
        destination=output_dir,
        options=options,
    )

    for r in download_results:
        status = "OK" if r.success else f"FAILED: {r.error}"
        print(f"  {r.data_id}: {status}")

### Extracting the VV measurement TIFF from the downloaded .SAFE archive

Sentinel-1 SLC products are delivered as a `.SAFE` folder (inside a `.zip`)
containing **6 separate measurement TIFFs** — one per sub-swath (IW1/IW2/IW3)
per polarisation (VV/VH). `InterferogramGenerator` needs a single flat
GeoTIFF per scene, so we need to:

1. Open the downloaded zip and list the measurement TIFFs
2. For each VV sub-swath, read its embedded GCPs (Sentinel-1 SLC TIFFs carry
   Ground Control Points that `rasterio` can read directly — no need to
   parse the annotation XML for a rough coverage check)
3. Pick the sub-swath whose GCP-derived footprint actually overlaps our AOI
4. Extract just that one TIFF to disk as a flat, usable file


In [ ]:
import zipfile

import numpy as np
import rasterio


def find_vv_subswaths(zip_path):
    """List the VV measurement TIFF entries inside an S1 SLC .SAFE zip."""
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
    vv_tiffs = [n for n in names if "/measurement/" in n and "-vv-" in n and n.endswith(".tiff")]
    return sorted(vv_tiffs)


def gcp_bbox(zip_path, member_name):
    """Read embedded GCPs from a zipped measurement TIFF and return its
    approximate (min_lon, min_lat, max_lon, max_lat) footprint."""
    vsi_path = f"/vsizip/{zip_path}/{member_name}"
    with rasterio.open(vsi_path) as src:
        gcps, gcp_crs = src.gcps
        if not gcps:
            return None
        lons = [g.x for g in gcps]
        lats = [g.y for g in gcps]
        return (min(lons), min(lats), max(lons), max(lats))


def bbox_overlaps(a, b):
    """Simple bbox overlap test: a, b = (min_lon, min_lat, max_lon, max_lat)."""
    return not (a[2] < b[0] or b[2] < a[0] or a[3] < b[1] or b[3] < a[1])


def extract_matching_subswath(zip_path, aoi_bbox, output_dir, label=""):
    """
    Find the VV sub-swath covering aoi_bbox and extract it as a flat .tif.

    Args:
        zip_path:   Path to the downloaded .SAFE.zip
        aoi_bbox:   BoundingBox object (from pygeofetch.models.search_query)
        output_dir: Where to write the extracted flat TIFF
        label:      "reference" or "secondary" for the output filename

    Returns:
        Path to the extracted flat GeoTIFF, or None if no sub-swath matched.
    """
    aoi = (aoi_bbox.min_lon, aoi_bbox.min_lat, aoi_bbox.max_lon, aoi_bbox.max_lat)
    vv_tiffs = find_vv_subswaths(zip_path)
    print(f"  Found {len(vv_tiffs)} VV sub-swath(s) in {Path(zip_path).name}")

    matched_member = None
    for member in vv_tiffs:
        footprint = gcp_bbox(zip_path, member)
        if footprint is None:
            continue
        swath = "IW1" if "iw1" in member else "IW2" if "iw2" in member else "IW3" if "iw3" in member else "?"
        overlaps = bbox_overlaps(footprint, aoi)
        print(f"    {swath}: footprint={tuple(round(v,2) for v in footprint)}  overlaps AOI: {overlaps}")
        if overlaps and matched_member is None:
            matched_member = member

    if matched_member is None:
        print("  No sub-swath overlaps the AOI — check your bbox or scene selection.")
        return None

    out_path = Path(output_dir) / f"{label}_vv.tif"
    with zipfile.ZipFile(zip_path) as zf:
        with zf.open(matched_member) as src_f, open(out_path, "wb") as dst_f:
            dst_f.write(src_f.read())

    print(f"  Extracted -> {out_path.name}")
    return out_path


if reference_scene is not None:
    ref_zip = output_dir / f"{reference_scene.id}.zip"
    sec_zip = output_dir / f"{secondary_scene.id}.zip"

    if ref_zip.exists() and sec_zip.exists():
        print("Reference scene:")
        reference_tif = extract_matching_subswath(ref_zip, aoi, output_dir, label="reference")
        print()
        print("Secondary scene:")
        secondary_tif = extract_matching_subswath(sec_zip, aoi, output_dir, label="secondary")
    else:
        print("Zip files not found at expected paths:")
        print(f"  {ref_zip}  exists={ref_zip.exists()}")
        print(f"  {sec_zip}  exists={sec_zip.exists()}")
        print("Check the actual downloaded filename in ./data/insar_pair/ and")
        print("adjust ref_zip/sec_zip above to match.")
        reference_tif = secondary_tif = None
else:
    reference_tif = secondary_tif = None
    print("No scenes to extract — search/download was skipped earlier.")


### Fetching precise orbit files

InSAR requires precise satellite position knowledge — the predicted orbit
embedded in the SLC product is not accurate enough. ESA publishes **Precise
Orbit Ephemerides (POEORB)** with ~20 days latency, and faster
**Restituted Orbits (RESORB)** for recent acquisitions.

PyGeoFetch's `orbits` module handles this automatically, including caching.


In [ ]:

if reference_scene is None:
    print("Skipping orbit file fetch — no scenes found in the search above.")
else:
    for scene in [reference_scene, secondary_scene]:
        try:
            orbit_path = client.fetch_orbit_file(scene.id)
            print(f"  {scene.id}: orbit file -> {orbit_path.name}")
        except Exception as exc:
            print(f"  {scene.id}: could not fetch orbit file ({exc})")
            print("    -> Will proceed with the orbit state vectors embedded in the SLC metadata.")

## Part 2 — Radiometric Calibration & Speckle Filtering

Before forming an interferogram, it's common practice to inspect and
optionally despeckle the individual SLC amplitude for quality control
(this does NOT affect the phase used for interferometry — despeckling
is applied to a separate calibrated amplitude product, not the raw SLC).

We calibrate to Sigma-nought (σ⁰) backscatter, the standard normalized
radar cross-section used for land-cover and change analysis.


In [ ]:
from pygeofetch.processing.sar import SARProcessor

sar = SARProcessor()

# reference_tif and secondary_tif were set by the extraction step above —
# they now point to the real, extracted, VV-only flat GeoTIFFs.

if reference_tif is not None and reference_tif.exists():
    calib_result = sar.calibrate(
        str(reference_tif),
        output_type="sigma0",
        in_db=True,
        output=str(output_dir / "reference_sigma0.tif"),
    )
    print(f"Calibration: {calib_result}")

    despeckle_result = sar.despeckle(
        str(output_dir / "reference_sigma0.tif"),
        filter="enhanced_lee",   # Lee (1981) refinement — better edge preservation
        window=5,
        output=str(output_dir / "reference_sigma0_despeckled.tif"),
    )
    print(f"Despeckle: {despeckle_result}")
else:
    print("Skipping calibration demo — no extracted TIFF available.")
    print("Check the extraction step above ran successfully.")


## Part 3 — Interferogram Generation

This is the core InSAR step: coregister the two SLC images, form the
interferogram by complex conjugate multiplication, and remove the
topographic phase component using a reference DEM.

**Coregistration accuracy matters enormously for TOPS mode.** Sentinel-1's
azimuth-varying Doppler centroid requires sub-0.001-pixel coregistration
accuracy to avoid visible phase discontinuities at burst edges — this is
why `InterferogramGenerator` applies Enhanced Spectral Diversity (ESD)
refinement on top of the initial geometric coregistration.


In [ ]:
from pygeofetch.insar import InterferogramGenerator

# A DEM is required for meaningful topographic phase removal.
# Options: SRTM 30m (global, free), Copernicus DEM 30m (newer, more accurate),
# or any GeoTIFF DEM covering your AOI in the same CRS as the SLC data.
dem_path = output_dir / "dem.tif"  # supply your own DEM here

gen = InterferogramGenerator(
    coherence_window=5,   # pixels — larger window = smoother coherence, less spatial detail
    esd_enabled=True,     # apply Enhanced Spectral Diversity refinement
)

if reference_tif is not None and secondary_tif is not None and reference_tif.exists() and secondary_tif.exists():
    result = gen.process_pair(
        reference=reference_tif,
        secondary=secondary_tif,
        dem=dem_path if dem_path.exists() else None,
        reference_date=str(reference_scene.datetime)[:10],
        secondary_date=str(secondary_scene.datetime)[:10],
    )

    print(f"Interferogram shape: {result.interferogram.shape}")
    print(f"Mean coherence: {result.coherence.mean():.3f}")
    print(f"ESD azimuth shift applied: {result.esd_azimuth_shift_px}")
    print(f"Metadata: {result.metadata}")
else:
    print("Run this cell after Part 2's extraction step produces the input TIFFs.")


In [ ]:
# Visualise the wrapped interferogram and coherence
import matplotlib.pyplot as plt

if 'result' in dir() and reference_tif.exists():
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    im0 = axes[0].imshow(np.angle(result.interferogram), cmap="twilight", vmin=-np.pi, vmax=np.pi)
    axes[0].set_title("Wrapped interferometric phase")
    plt.colorbar(im0, ax=axes[0], label="radians", fraction=0.046)

    im1 = axes[1].imshow(result.coherence, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("Interferometric coherence")
    plt.colorbar(im1, ax=axes[1], label="coherence (0-1)", fraction=0.046)

    plt.tight_layout()
    plt.savefig("interferogram_preview.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("\nSaved: interferogram_preview.png")


In [ ]:
# Save the interferogram products to disk
if 'result' in dir() and reference_tif.exists():
    saved_paths = result.save(output_dir / "interferogram")
    for name, path in saved_paths.items():
        print(f"  {name}: {path}")


## Part 4 — Phase Unwrapping

The wrapped phase is only known modulo 2π. **Phase unwrapping** recovers
the continuous phase signal needed to compute actual displacement.

PyGeoFetch uses SNAPHU (Statistical-cost, Network-flow Algorithm for Phase
Unwrapping) via `snaphu-py` — the exact same algorithm used in ASF's
On Demand InSAR products and bundled with ISCE2/3.

**Cost mode choice matters:**
- `"defo"` — for deformation monitoring (subsidence, earthquakes, volcanoes).
  Less smoothing bias than `"topo"`, recommended for SBAS/PS time series.
- `"topo"` — for topographic mapping / DEM generation, where phase
  gradients are expected to follow terrain.


In [ ]:
from pygeofetch.insar import PhaseUnwrapper

unwrapper = PhaseUnwrapper(
    cost_mode="defo",     # deformation-optimized cost function
    init_method="mcf",    # Minimum Cost Flow — matches ASF/GAMMA convention
)

if 'result' in dir() and reference_tif.exists():
    unwrapped, conncomp = unwrapper.unwrap(
        result.interferogram,
        result.coherence,
        nlooks=4.0,   # match any multilooking applied during interferogram formation
    )

    n_reliable = np.sum(conncomp > 0)
    pct_reliable = 100 * n_reliable / conncomp.size
    print(f"Unwrapped phase range: [{unwrapped.min():.2f}, {unwrapped.max():.2f}] radians")
    print(f"Reliable pixels (conncomp > 0): {pct_reliable:.1f}%")


In [ ]:
# Save the unwrapped result and visualise
if 'unwrapped' in dir():
    saved_unw = unwrapper.unwrap_files(
        saved_paths["wrapped_phase"],
        saved_paths["coherence"],
        output_path=output_dir / "interferogram" / "unwrapped.tif",
        nlooks=4.0,
    )

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(unwrapped, cmap="RdBu_r")
    ax.set_title("Unwrapped phase")
    plt.colorbar(im, ax=ax, label="radians", fraction=0.046)
    plt.tight_layout()
    plt.savefig("unwrapped_preview.png", dpi=120, bbox_inches="tight")
    plt.show()


## Part 5 — Convert Phase to Displacement

Line-of-sight (LOS) displacement is derived from unwrapped phase using
the radar wavelength:

$$d_{LOS} = -\frac{\lambda}{4\pi} \cdot \phi_{unwrapped}$$

For Sentinel-1 C-band, λ = 0.05546576 m. The negative sign follows the
convention that increasing phase corresponds to the target moving *away*
from the satellite (subsidence on a typical descending pass).


In [ ]:
SENTINEL1_WAVELENGTH_M = 0.05546576  # C-band

if 'unwrapped' in dir():
    displacement_m = -unwrapped * SENTINEL1_WAVELENGTH_M / (4 * np.pi)
    displacement_mm = displacement_m * 1000

    print(f"LOS displacement range: [{displacement_mm.min():.1f}, {displacement_mm.max():.1f}] mm")
    print("(single interferometric pair — this is the displacement between the two acquisition dates)")

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(displacement_mm, cmap="RdBu_r", vmin=-50, vmax=50)
    ax.set_title(f"LOS displacement ({reference_scene.datetime} to {secondary_scene.datetime})")
    plt.colorbar(im, ax=ax, label="mm", fraction=0.046)
    plt.tight_layout()
    plt.show()


## Part 6 — Atmospheric Correction

Tropospheric delay is one of the dominant error sources in InSAR,
typically contributing 2-10 cm of apparent "signal" that is actually
atmospheric noise, not ground motion.

Two methods are available:
- **Elevation-correlated** (native, no extra dependencies): regresses phase
  against DEM elevation to remove the dominant stratified atmospheric
  component. Works well in areas with limited turbulent mixing.
- **ERA5-based** (`pip install "pygeofetch[insar-full]"`): computes the
  actual tropospheric zenith delay from ECMWF reanalysis data and projects
  it along the line-of-sight. More accurate but requires CDS API credentials
  and network access.


In [ ]:
from pygeofetch.insar import AtmosphericCorrector

if 'unwrapped' in dir() and dem_path.exists():
    corrector = AtmosphericCorrector(method="elevation")
    corrected_phase = corrector.correct(unwrapped, dem=dem_path)

    displacement_corrected_mm = -corrected_phase * SENTINEL1_WAVELENGTH_M / (4 * np.pi) * 1000

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    im0 = axes[0].imshow(displacement_mm, cmap="RdBu_r", vmin=-50, vmax=50)
    axes[0].set_title("Before atmospheric correction")
    plt.colorbar(im0, ax=axes[0], fraction=0.046, label="mm")

    im1 = axes[1].imshow(displacement_corrected_mm, cmap="RdBu_r", vmin=-50, vmax=50)
    axes[1].set_title("After elevation-correlated correction")
    plt.colorbar(im1, ax=axes[1], fraction=0.046, label="mm")
    plt.tight_layout()
    plt.show()
else:
    print("Requires a DEM — see Part 3 for DEM setup.")


## Part 7 — Multi-Pair Time Series (SBAS)

A single interferometric pair gives displacement between two dates. For a
**time series** of ground motion, you need a network of interferograms
spanning multiple dates and invert them jointly using the Small BAseline
Subset (SBAS) method (Berardino et al. 2002).

This section shows the workflow structure using synthetic pairs — in a
real project, you would repeat Parts 3-6 for every pair in your network
and feed all the results into `SBASTimeSeries`.


In [ ]:

# In a real project, build this list by running Parts 3-6 for every
# interferometric pair in your SBAS network (e.g. all pairs with
# temporal baseline < 48 days and perpendicular baseline < 150m).
#
# Structure:
#   pairs = [
#       InterferogramPair(date1, date2, unwrapped_phase_array, coherence_array),
#       InterferogramPair(date2, date3, unwrapped_phase_array, coherence_array),
#       ...
#   ]

print("SBAS network construction pattern:")
print()
print("  pairs = []")
print("  for (date_a, date_b) in your_pair_list:")
print("      result = gen.process_pair(slc[date_a], slc[date_b], dem=dem_path)")
print("      unw, _ = unwrapper.unwrap(result.interferogram, result.coherence)")
print("      pairs.append(InterferogramPair(date_a, date_b, unw, result.coherence))")
print()
print("  sbas = SBASTimeSeries(wavelength_m=0.05546576)")
print("  ts_result = sbas.invert(pairs)")
print()
print("See the companion notebook '02_insar_subsidence_monitoring.ipynb'")
print("for a complete, runnable end-to-end SBAS time series example.")


## Summary

You've now walked through the complete PyGeoFetch InSAR chain:

1. ✅ **Search & download** SLC pairs with automatic provider routing
2. ✅ **Precise orbit files** for accurate satellite geometry
3. ✅ **Radiometric calibration** (σ⁰) and **speckle filtering**
4. ✅ **Interferogram generation** with ESD coregistration and topographic phase removal
5. ✅ **Phase unwrapping** via SNAPHU (production-grade, same algorithm as ASF/ISCE/SNAP)
6. ✅ **Displacement conversion** from unwrapped phase
7. ✅ **Atmospheric correction** (elevation-correlated and ERA5-based)
8. ✅ **SBAS time series** structure for multi-date deformation monitoring

**Next:** see `02_insar_subsidence_monitoring.ipynb` for a complete,
real-world project applying this entire chain to groundwater-extraction
subsidence monitoring with a full SBAS time series and velocity map.

### References

- Chen, C.W. & Zebker, H.A. (2001). *Two-dimensional phase unwrapping with
  use of statistical models for cost functions in a network programming
  framework.* JOSA A, 18(2), 338-351.
- Berardino, P. et al. (2002). *A new algorithm for surface deformation
  monitoring based on small baseline differential SAR interferograms.*
  IEEE TGRS, 40(11), 2375-2383.
- Scheiber, R. & Moreira, A. (2000). *Coregistration of interferometric
  SAR images using spectral diversity.* IEEE TGRS, 38(5), 2179-2191.
- Yunjun, Z., Fattahi, H., & Amelung, F. (2019). *Small baseline InSAR time
  series analysis: unwrapping error correction and noise reduction.*
  Computers & Geosciences, 133, 104331.
- Jolivet, R. et al. (2011). *Systematic InSAR tropospheric phase delay
  corrections from global meteorological reanalysis data.* GRL, 38(17).
